# Geographic Scalars for Solar (SSRD)

This tutorial shows how loss scalars on the ERA5 0.25° grid are generated from **operating utility-scale solar** capacity. The same scalar field is intended for `surface_solar_radiation_downwards` (SSRD). A separate capacity-weighted field is used for 100 m wind u/v; temperature may share the solar field or use its own weights depending on deployment.


## Data sources

1. **Global Solar Power Tracker** (Global Energy Monitor, February 2026) — utility-scale (≥1 MW) plant locations and `Capacity (MW)`.
2. **Natural Earth** — country polygons used to build buffered Europe and Germany masks.
3. **ERA5 0.25° grid** — 721 latitudes in `[-90, 90]` and 1440 longitudes in `[-180, 179.75]`.

### Getting the solar data

1. Navigate to the [Global Solar Power Tracker](https://globalenergymonitor.org/projects/global-solar-power-tracker).
2. Download the latest Excel release.
3. Save it next to this notebook as `Global-Solar-Power-Tracker-February-2026.xlsx`.
4. This notebook uses the **`Utility-Scale (1 MW+)`** sheet and keeps all rows with `Status == "operating"` (no AC/DC filter).

> **Citation:** *"Global Solar Power Tracker, Global Energy Monitor, February 2026 release."*


## Setup

Install the packages needed to run this notebook by itself.


In [ ]:
%pip install -q pandas openpyxl numpy scipy geopandas shapely matplotlib cartopy requests


In [1]:
import io
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import scipy.ndimage as ndimage
from matplotlib.colors import LogNorm
from shapely.geometry import Point
from shapely.prepared import prep

CANDIDATES = [
    Path.cwd(),
    Path.cwd() / "notebooks",
    Path("/root/Zeus/notebooks"),
]
NOTEBOOK_DIR = next(
    (p for p in CANDIDATES if (p / "Global-Solar-Power-Tracker-February-2026.xlsx").exists()),
    Path("/root/Zeus/notebooks"),
)
EXPORT_DIR = NOTEBOOK_DIR
print("Notebook / data directory:", NOTEBOOK_DIR.resolve())


Notebook / data directory: /root/Zeus/notebooks


## 1. Load operating utility-scale solar plants

Keep only plants with status `operating` from the utility-scale sheet, rename the columns used later, and drop rows without coordinates or capacity.


In [2]:
EXCEL_PATH = NOTEBOOK_DIR / "Global-Solar-Power-Tracker-February-2026.xlsx"

df = pd.read_excel(EXCEL_PATH, sheet_name="Utility-Scale (1 MW+)")
df = df[df["Status"] == "operating"].copy()
df = df.rename(
    columns={
        "Capacity (MW)": "capacity",
        "Latitude": "lat",
        "Longitude": "lon",
        "Region": "region",
        "Country/Area": "country",
    }
)
df["capacity"] = pd.to_numeric(df["capacity"], errors="coerce")
df = df.dropna(subset=["lat", "lon", "capacity"])

print(f"Operating utility-scale solar plants: {len(df)}")
print(f"Lat range: {df['lat'].min():.4f} .. {df['lat'].max():.4f}")
print(f"Lon range: {df['lon'].min():.4f} .. {df['lon'].max():.4f}")
df[["country", "capacity", "lat", "lon", "region"]].head()


Operating utility-scale solar plants: 81377
Lat range: -44.4482 .. 68.3784
Lon range: -175.3292 .. 177.9816


,country,capacity,lat,lon,region
6,Afghanistan,13.9,31.6265,65.8542,Asia
22,Afghanistan,1.5,34.1829,62.2165,Asia
28,Afghanistan,1.6,34.4151,70.4034,Asia
33,Afghanistan,12.0,31.4650,65.8804,Asia
34,Afghanistan,12.0,31.4630,65.8750,Asia


## 2. ERA5 grid and region masks

Building the region bases using Natural Earth geometries with a **2° buffer**. The scalers ae then dependent on the capacity. 

In [3]:
GEOJSON_URL = (
    "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/"
    "geojson/ne_50m_admin_0_countries.geojson"
)

N_LAT = 721
N_LON = 1440
GRID_RES = 0.25
BUFFER_DEG = 2.0

EUROPE_LAT_MIN, EUROPE_LAT_MAX = 27.5, 72.25
EUROPE_LON_MIN, EUROPE_LON_MAX = -25.50, 45.25
GERMANY_LAT_MIN, GERMANY_LAT_MAX = 47.0, 56.25
GERMANY_LON_MIN, GERMANY_LON_MAX = 5.25, 15.25


def make_era5_grid():
    """0.25° grid: latitude south→north, longitude in [-180, 180)."""
    lats = np.linspace(-90.0, 90.0, N_LAT)
    lons = np.linspace(-180.0, 179.75, N_LON)
    return lats, lons


def fetch_country_geometries():
    headers = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64)"}
    response = requests.get(GEOJSON_URL, headers=headers, timeout=30)
    response.raise_for_status()
    return gpd.read_file(io.BytesIO(response.content))


def generate_region_masks(lats, lons, buffer_deg=BUFFER_DEG):
    gdf = fetch_country_geometries()

    germany_geom = gdf[gdf["ADMIN"] == "Germany"].geometry.union_all()
    europe_geom = gdf[gdf["CONTINENT"] == "Europe"].geometry.union_all()
    russia_geom = gdf[gdf["ADMIN"] == "Russia"].geometry.union_all()
    europe_geom = europe_geom.difference(russia_geom)

    europe_buffered = europe_geom.buffer(buffer_deg)
    germany_buffered = germany_geom.buffer(buffer_deg)

    prep_eu = prep(europe_buffered)
    prep_de = prep(germany_buffered)

    grid_lon, grid_lat = np.meshgrid(lons, lats)
    flat_lons = grid_lon.flatten()
    flat_lats = grid_lat.flatten()

    de_mask_flat = np.zeros(len(flat_lats), dtype=bool)
    eu_mask_flat = np.zeros(len(flat_lats), dtype=bool)

    eu_candidate_idx = np.where(
        (flat_lats >= EUROPE_LAT_MIN)
        & (flat_lats < EUROPE_LAT_MAX)
        & (flat_lons >= EUROPE_LON_MIN)
        & (flat_lons < EUROPE_LON_MAX)
    )[0]

    de_candidate_idx = np.where(
        (flat_lats >= GERMANY_LAT_MIN)
        & (flat_lats < GERMANY_LAT_MAX)
        & (flat_lons >= GERMANY_LON_MIN)
        & (flat_lons < GERMANY_LON_MAX)
    )[0]

    for idx in eu_candidate_idx:
        if prep_eu.contains(Point(flat_lons[idx], flat_lats[idx])):
            eu_mask_flat[idx] = True

    for idx in de_candidate_idx:
        if prep_de.contains(Point(flat_lons[idx], flat_lats[idx])):
            de_mask_flat[idx] = True

    # Germany wins over Europe where they overlap
    eu_mask_flat[de_mask_flat] = False

    de_mask = de_mask_flat.reshape((len(lats), len(lons)))
    eu_mask = eu_mask_flat.reshape((len(lats), len(lons)))
    row_mask = ~(de_mask | eu_mask)
    return de_mask, eu_mask, row_mask


lats, lons = make_era5_grid()
de_mask, eu_mask, row_mask = generate_region_masks(lats, lons)

print(f"Grid: {len(lats)} x {len(lons)} (res={GRID_RES}°)")
print(f"Germany cells: {int(de_mask.sum())}")
print(f"Europe (ex-Germany) cells: {int(eu_mask.sum())}")
print(f"Rest of world cells: {int(row_mask.sum())}")


Grid: 721 x 1440 (res=0.25°)
Germany cells: 1447
Europe (ex-Germany) cells: 23483
Rest of world cells: 1013310


## 3. Capacity → scalars

Pipeline:

1. **Bin** each plant's MW onto the nearest 0.25° cell.
2. **Smooth** with a Gaussian filter (`sigma=1.5` grid cells, `mode="wrap"` so longitude wraps).
3. **Assign regional budgets**: Germany **0.40**, rest of Europe **0.40**, rest of world **0.20**.

For rest of world, every cell gets the same value `0.20 / n_row`.

For Germany and Europe, each cell first receives that same ROW baseline. The leftover regional budget is then distributed **in proportion to smoothed solar capacity** inside the region:

```text
scalar[cell] = row_baseline + capacity_budget * (smoothed_capacity[cell] / sum(smoothed_capacity in region))
```

Finally rescale so the global minimum is **1** (`scalars / scalars.min()`). Relative ratios are unchanged.


In [4]:
def _grid_indices(solar_df, lats, lons):
    dlat = float(lats[1] - lats[0])
    dlon = float(lons[1] - lons[0])
    lat_indices = np.clip(
        np.round((solar_df["lat"].values - lats[0]) / dlat).astype(int),
        0,
        len(lats) - 1,
    )
    lon_indices = np.clip(
        np.round((solar_df["lon"].values - lons[0]) / dlon).astype(int),
        0,
        len(lons) - 1,
    )
    return lat_indices, lon_indices


def bin_capacity_to_grid(solar_df, lats, lons):
    """Sum plant capacity onto the nearest 0.25° cell."""
    raw = np.zeros((len(lats), len(lons)), dtype=np.float64)
    lat_i, lon_i = _grid_indices(solar_df, lats, lons)
    np.add.at(raw, (lat_i, lon_i), solar_df["capacity"].values)
    return raw


def compute_global_scalar_matrix(
    solar_df, lats, lons, de_mask, eu_mask, row_mask, sigma_grid_cells=1.5
):
    raw_capacity_grid = bin_capacity_to_grid(solar_df, lats, lons)
    smoothed_grid = ndimage.gaussian_filter(
        raw_capacity_grid, sigma=sigma_grid_cells, mode="wrap"
    )

    num_row_cells = np.count_nonzero(row_mask)
    row_cell_value = 0.20 / num_row_cells

    scalars = np.zeros(raw_capacity_grid.shape, dtype=np.float64)
    scalars[row_mask] = row_cell_value

    def _assign_regional_scalars(mask, regional_total):
        count = int(np.count_nonzero(mask))
        if count == 0:
            return
        baseline_total = count * row_cell_value
        capacity_budget = regional_total - baseline_total
        capacity_signal = smoothed_grid[mask]
        capacity_sum = float(capacity_signal.sum())
        if capacity_budget <= 0.0 or capacity_sum <= 0.0:
            scalars[mask] = regional_total / count
            return
        scalars[mask] = row_cell_value + capacity_budget * (capacity_signal / capacity_sum)

    _assign_regional_scalars(de_mask, 0.40)
    _assign_regional_scalars(eu_mask, 0.40)
    return scalars, raw_capacity_grid, smoothed_grid


def normalize_scalars_min_one(scalars):
    scalar_min = float(np.min(scalars))
    if scalar_min <= 0.0:
        raise ValueError(f"Cannot normalize non-positive scalars, min={scalar_min}")
    return scalars / scalar_min


scalars_raw, capacity_grid, smoothed_capacity = compute_global_scalar_matrix(
    df, lats, lons, de_mask, eu_mask, row_mask
)
scalars = normalize_scalars_min_one(scalars_raw)

print(f"Unnormalized sum (should be ~1): {scalars_raw.sum():.6f}")
print(f"Normalized min/max: {scalars.min():.4f} / {scalars.max():.4f}")
print(f"Capacity cells with MW > 0: {int((capacity_grid > 0).sum())}")


Unnormalized sum (should be ~1): 1.000000
Normalized min/max: 1.0000 / 5355.1611
Capacity cells with MW > 0: 17477


## 4. Region statistics

Stats use the Europe / Germany lat–lon boxes (same candidate boxes as the mask step). Europe here means the Europe box **excluding** the Germany box; rest of world is everything outside the Europe box.


In [5]:
def _bbox_mask(lats, lons, lat_min, lat_max, lon_min, lon_max):
    lat_sel = (lats >= lat_min) & (lats < lat_max)
    lon_sel = (lons >= lon_min) & (lons < lon_max)
    return lat_sel[:, None] & lon_sel[None, :]


def print_region_stats(values, lats, lons, label):
    europe_bbox = _bbox_mask(
        lats, lons, EUROPE_LAT_MIN, EUROPE_LAT_MAX, EUROPE_LON_MIN, EUROPE_LON_MAX
    )
    germany_mask = _bbox_mask(
        lats, lons, GERMANY_LAT_MIN, GERMANY_LAT_MAX, GERMANY_LON_MIN, GERMANY_LON_MAX
    )
    regions = {
        "Europe": europe_bbox & ~germany_mask,
        "Germany": germany_mask,
        "Rest of world": ~europe_bbox,
    }
    print(f"\n{label}")
    for name, mask in regions.items():
        vals = values[mask]
        print(
            f"  {name}: cells={int(mask.sum())} "
            f"min={float(vals.min()):.6e} "
            f"max={float(vals.max()):.6e} "
            f"mean={float(vals.mean()):.6e} "
            f"sum={float(vals.sum()):.6e}"
        )
    print(f"  Total sum: {float(values.sum()):.6e}")


print_region_stats(scalars_raw, lats, lons, "Scalar stats (unnormalized)")
print_region_stats(scalars, lats, lons, "Scalar stats (normalized min=1)")



Scalar stats (unnormalized)
  Europe: cells=49177 min=1.973730e-07 max=6.948712e-04 mean=8.233534e-06 sum=4.049005e-01
  Germany: cells=1480 min=1.973730e-07 max=1.056964e-03 mean=2.703901e-04 sum=4.001773e-01
  Rest of world: cells=987583 min=1.973730e-07 max=1.973730e-07 mean=1.973730e-07 sum=1.949222e-01
  Total sum: 1.000000e+00

Scalar stats (normalized min=1)
  Europe: cells=49177 min=1.000000e+00 max=3.520600e+03 mean=4.171561e+01 sum=2.051449e+06
  Germany: cells=1480 min=1.000000e+00 max=5.355161e+03 mean=1.369945e+03 sum=2.027518e+06
  Rest of world: cells=987583 min=1.000000e+00 max=1.000000e+00 mean=1.000000e+00 sum=9.875830e+05
  Total sum: 5.066550e+06


## 5. Map of normalized scalars

Cartopy layout matches the earlier geographic-scalar maps. Values are continuous (capacity-weighted), so a **log** color scale is used.


In [6]:
LONS, LATS = np.meshgrid(lons, lats)

plt.style.use("default")
plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 14,
        "figure.titlesize": 16,
    }
)

fig = plt.figure(figsize=(12, 6))
ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=0))

# Set extent before drawing features / mesh so layout stays stable.
ax.set_extent([-80, 80, 0, 85], crs=ccrs.PlateCarree())

ax.add_feature(
    cfeature.NaturalEarthFeature(
        "physical", "land", "50m", edgecolor="face", facecolor="#f4f6f8"
    ),
    zorder=0,
)
ax.add_feature(
    cfeature.NaturalEarthFeature(
        "physical", "ocean", "50m", edgecolor="face", facecolor="#ffffff"
    ),
    zorder=0,
)
ax.add_feature(
    cfeature.BORDERS.with_scale("50m"),
    linewidth=0.3,
    edgecolor="#999999",
    linestyle="--",
    zorder=0,
)
ax.coastlines(resolution="50m", linewidth=0.5, color="#333333", zorder=1)

norm = LogNorm(vmin=float(scalars.min()), vmax=float(scalars.max()))
# 1D lon/lat is more reliable with cartopy than a full meshgrid.
img = ax.pcolormesh(
    lons,
    lats,
    scalars,
    transform=ccrs.PlateCarree(),
    cmap="YlOrRd",
    norm=norm,
    shading="auto",
    alpha=0.75,
    zorder=2,
)

cbar = plt.colorbar(img, ax=ax, orientation="vertical", pad=0.02, fraction=0.035, shrink=0.85)
cbar.set_label("Normalized scalar (log scale)", fontsize=11)

gl = ax.gridlines(
    draw_labels=True, linewidth=0.5, color="gray", alpha=0.3, linestyle=":"
)
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {"size": 9, "color": "#333333"}
gl.ylabel_style = {"size": 9, "color": "#333333"}

plt.title(
    "Geographic Scalars for Solar / SSRD (capacity-weighted, min = 1)",
    pad=20,
    weight="bold",
)

# bbox_inches="tight" drops the GeoAxes when gridline labels are enabled (colorbar-only PNG).
fig.canvas.draw()
png_path = EXPORT_DIR / "new_solar_geographic_scalars.png"
plt.savefig(png_path, dpi=300, pad_inches=0.2)
print(f"Exported to {png_path}")
plt.close()


Exported to /root/Zeus/notebooks/new_solar_geographic_scalars.png


In [7]:
# Save normalized scalars with the lat/lon axes used to build them
from pathlib import Path

npz_path = Path("/root/Zeus/zeus/data/weights") / "new_solar_scalars.npz"
npz_path.parent.mkdir(parents=True, exist_ok=True)
np.savez(npz_path, scalars=scalars, lats=lats, lons=lons)
print(f"Saved {npz_path}")
print(f"  scalars={scalars.shape} lats={lats.shape} lons={lons.shape}")
print(f"  lat [{lats[0]:.2f}, {lats[-1]:.2f}] lon [{lons[0]:.2f}, {lons[-1]:.2f}]")


Saved /root/Zeus/zeus/data/weights/new_solar_scalars.npz
  scalars=(721, 1440) lats=(721,) lons=(1440,)
  lat [-90.00, 90.00] lon [-180.00, 179.75]


> Note that it doesn't matter if we save/are using the normalised or unnormalised scalars as the the scoring applies normalisation after the weights are combined with the latitude ones. Check `zeus/validator/metrics.py` -> custom_rmse()